# Data QA Review — Smoke-Resilient Intruder Detection

**Cel:** Przejrzyj ~200 zdjęć, potwierdź że obraz ↔ opis się zgadzają.

**Skala oceny:**
- **5** — idealnie, dokładnie to co opisane
- **4** — OK, drobne niedokładności ale akceptowalne
- **3** — wątpliwe, lepiej sprawdzić
- **2** — poważny błąd (zła kategoria, błędna liczba osób)
- **1** — totalnie nie to (wieszak zamiast osoby, głupota)

**Workflow:** uruchom Cell 1, potem Cell 2. Cell 3 tylko na końcu do raportu.

In [1]:
# Cell 1: Setup — uruchom raz
import os
import csv
import glob
from pathlib import Path
from IPython.display import display
import ipywidgets as w
from PIL import Image
import io

PACK_DIR = Path('../review_pack')
RESULTS_FILE = Path('results.csv')
FIELDNAMES = ['filename', 'category', 'rating', 'flagged', 'comment']

# Załaduj wszystkie pary (jpg + txt)
all_images = sorted(glob.glob(str(PACK_DIR / '**/*.jpg'), recursive=True))
assert all_images, f'Brak zdjęć w {PACK_DIR} — sprawdź ścieżkę'
print(f'Znaleziono {len(all_images)} zdjęć w {len(set(Path(p).parent.name for p in all_images))} folderach')

# Załaduj dotychczasowe wyniki (resume)
reviewed = {}
if RESULTS_FILE.exists():
    with open(RESULTS_FILE, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            reviewed[row['filename']] = row
    print(f'Wczytano {len(reviewed)} wcześniejszych ocen — wznawiam od miejsca')
else:
    print('Nowa sesja — zaczynam od początku')

def save_result(filename, rating, flagged, comment):
    category = Path(filename).parent.name
    row = {'filename': filename, 'category': category,
           'rating': rating, 'flagged': str(flagged), 'comment': comment}
    reviewed[filename] = row
    write_header = not RESULTS_FILE.exists()
    with open(RESULTS_FILE, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if write_header:
            writer.writeheader()
        writer.writerow(row)

def load_txt(jpg_path):
    txt_path = Path(jpg_path).with_suffix('.txt')
    return txt_path.read_text(encoding='utf-8').strip() if txt_path.exists() else '(brak opisu)'

def img_to_bytes(jpg_path, max_width=500):
    img = Image.open(jpg_path).convert('RGB')
    ratio = max_width / img.width
    img = img.resize((max_width, int(img.height * ratio)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return buf.getvalue()

print('Setup OK — uruchom Cell 2')

Znaleziono 200 zdjęć w 6 folderach
Nowa sesja — zaczynam od początku
Setup OK — uruchom Cell 2


In [2]:
# Cell 2: Interfejs review

unreviewed = [p for p in all_images if p not in reviewed]
start_idx = all_images.index(unreviewed[0]) if unreviewed else 0
state = {'idx': start_idx}

# Widgety
img_widget   = w.Image(format='jpeg', width=500)
info_widget  = w.HTML(layout=w.Layout(width='400px'))
comment_box  = w.Text(placeholder='Komentarz opcjonalny…', layout=w.Layout(width='500px'))
progress_bar = w.IntProgress(min=0, max=len(all_images), description='Progress:',
                              style={'bar_color': '#27ae60'}, layout=w.Layout(width='500px'))
status_label = w.HTML()
flag_btn     = w.ToggleButton(description='🚩 Flag', button_style='warning',
                               layout=w.Layout(width='90px', height='36px'))

btn_layout = w.Layout(width='56px', height='36px')
rating_btns = [
    w.Button(description=str(i), layout=btn_layout,
             button_style='danger' if i <= 2 else ('warning' if i == 3 else 'success'))
    for i in range(1, 6)
]
prev_btn = w.Button(description='← Prev', layout=w.Layout(width='80px', height='36px'))
skip_btn = w.Button(description='Skip →', layout=w.Layout(width='80px', height='36px'))

CAT_COLORS = {
    'A_real_test': '#c0392b', 'B_real_test': '#2980b9', 'C_real_test': '#8e44ad',
    'A_train_sample': '#e67e22', 'B_train_sample': '#27ae60', 'C_train_sample': '#16a085'
}

def render(idx):
    path = all_images[idx]
    cat  = Path(path).parent.name
    txt  = load_txt(path)
    rel  = str(Path(path).relative_to(PACK_DIR.parent))

    img_widget.value = img_to_bytes(path)

    prev = reviewed.get(path, {})
    prev_badge = ''
    if prev:
        r = prev.get('rating', '?')
        f = ' 🚩' if prev.get('flagged') == 'True' else ''
        prev_badge = f'<span style="background:#e74c3c;color:white;padding:1px 6px;border-radius:3px;margin-left:6px">★{r}{f}</span>'

    color = CAT_COLORS.get(cat, '#555')
    info_widget.value = f"""
<div style="font-family:monospace;font-size:12px;padding:10px;background:#1e1e2e;
            color:#cdd6f4;border-radius:6px;line-height:1.6">
  <div style="margin-bottom:6px">
    <span style="background:{color};color:white;padding:2px 8px;border-radius:4px;font-weight:bold">{cat}</span>
    <span style="color:#a6adc8;margin-left:6px">{idx+1} / {len(all_images)}</span>
    {prev_badge}
  </div>
  <hr style="border-color:#313244;margin:6px 0">
  <pre style="white-space:pre-wrap;margin:0;font-size:11px;color:#cdd6f4">{txt}</pre>
  <hr style="border-color:#313244;margin:6px 0">
  <span style="color:#585b70;font-size:10px">{rel}</span>
</div>"""

    rejected = sum(1 for r in reviewed.values() if int(r.get('rating', 5)) < 4)
    flagged_n = sum(1 for r in reviewed.values() if r.get('flagged') == 'True')
    progress_bar.value = len(reviewed)
    status_label.value = (
        f"<b>{len(reviewed)}</b>/{len(all_images)} ocenionych &nbsp;·&nbsp; "
        f"<span style='color:#e74c3c'><b>{rejected}</b> odrzucone (&lt;4)</span> &nbsp;·&nbsp; "
        f"<span style='color:#f39c12'><b>{flagged_n}</b> flagowane</span>"
    )
    flag_btn.value = prev.get('flagged') == 'True'
    comment_box.value = prev.get('comment', '')

def rate(rating):
    path = all_images[state['idx']]
    save_result(path, rating, flag_btn.value, comment_box.value)
    comment_box.value = ''
    flag_btn.value = False
    # Skocz do następnego nieprzejrzanego
    nxt = state['idx'] + 1
    while nxt < len(all_images) and all_images[nxt] in reviewed:
        nxt += 1
    state['idx'] = min(nxt, len(all_images) - 1)
    render(state['idx'])

for btn in rating_btns:
    btn.on_click(lambda b: rate(int(b.description)))

prev_btn.on_click(lambda _: (state.update(idx=max(0, state['idx'] - 1)), render(state['idx'])))
skip_btn.on_click(lambda _: (state.update(idx=min(len(all_images)-1, state['idx']+1)), render(state['idx'])))

rating_row = w.HBox([prev_btn, *rating_btns, flag_btn, skip_btn],
                    layout=w.Layout(gap='4px', align_items='center', margin='6px 0'))
left  = w.VBox([img_widget, rating_row, comment_box])
right = w.VBox([info_widget])

display(w.VBox([status_label, progress_bar, w.HBox([left, right], layout=w.Layout(gap='12px'))]))
render(state['idx'])

In [ ]:
# Cell 3: Raport końcowy — uruchom po skończeniu review
import pandas as pd

if not RESULTS_FILE.exists():
    print('Brak results.csv — najpierw oceń przynajmniej jedno zdjęcie (Cell 2)')
else:
    df = pd.read_csv(RESULTS_FILE)
    df['rating'] = df['rating'].astype(int)
    df['flagged'] = df['flagged'].astype(str) == 'True'

    total = len(all_images)
    done  = len(df)
    ok    = (df.rating >= 4).sum()
    bad   = (df.rating < 4).sum()

    print('=== PODSUMOWANIE QA ===')
    print(f'Oceniono:       {done} / {total} ({done/total*100:.0f}%)')
    print(f'OK (≥4):        {ok} ({ok/done*100:.1f}%)')
    print(f'Odrzucone (<4): {bad} ({bad/done*100:.1f}%)')
    print(f'Flagowane:      {df.flagged.sum()}')
    print()

    print('--- Per kategoria ---')
    summary = df.groupby('category')['rating'].agg(
        n='count',
        avg='mean',
        pct_ok=lambda x: (x >= 4).mean() * 100
    ).round(1)
    print(summary.to_string())
    print()

    problems = df[(df.rating < 4) | df.flagged]
    if len(problems):
        print(f'--- Problemy ({len(problems)} zdjęć) ---')
        print(problems[['filename', 'category', 'rating', 'flagged', 'comment']]
              .to_string(index=False))
    else:
        print('Brak problemów!')

    # Verdict
    pct = ok / done * 100 if done else 0
    print()
    if pct >= 80:
        print(f'✅ GO — {pct:.1f}% OK (próg: 80%) → pipeline dobry, można skalować')
    else:
        print(f'❌ NO-GO — {pct:.1f}% OK (próg: 80%) → naprawić pipeline przed skalowaniem')